# 01b - Exploration de la bronze

exploration du csv bronze 

Ce notebook ne modifie rien : il  nous permet de lire le Bronze et prendre connaissance en profondeur de notre dataset

In [1]:
import sys
sys.path.append("/home/jovyan/work")

from pyspark.sql import functions as F

from outils.source import creer_session, LAKE

ANNEE = 2025
BRONZE = f"{LAKE}/bronze/dvf_{ANNEE}"

spark = creer_session("exploration")
df = spark.read.format("delta").load(BRONZE)
print(len(df.columns), "colonnes")

Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3. Le premier démarrage télécharge des jars : plusieurs minutes.
Session prête.
40 colonnes


## 1. Les valeurs de `nature_mutation`
Quelles natures de mutation existent, et combien de lignes / de mutations pour chacune ? 

In [2]:
(df.groupBy("nature_mutation")
   .agg(F.count("*").alias("lignes"), F.countDistinct("id_mutation").alias("mutations"))
   .orderBy(F.desc("mutations"))
   .show(truncate=False))

+----------------------------------+-------+---------+
|nature_mutation                   |lignes |mutations|
+----------------------------------+-------+---------+
|Vente                             |3585313|1309960  |
|Vente en l'état futur d'achèvement|143873 |51982    |
|Echange                           |45710  |13277    |
|Vente terrain à bâtir             |8116   |4653     |
|Adjudication                      |8060   |2665     |
|Expropriation                     |2041   |1817     |
+----------------------------------+-------+---------+



## 2. Les valeurs de `type_local` (vide = terrain sans local)

In [3]:
(df.groupBy("type_local")
   .agg(F.count("*").alias("lignes"), F.countDistinct("id_mutation").alias("mutations"))
   .orderBy(F.desc("lignes"))
   .show(truncate=False))

+----------------------------------------+-------+---------+
|type_local                              |lignes |mutations|
+----------------------------------------+-------+---------+
|NULL                                    |1497725|544751   |
|Dépendance                              |1018269|578766   |
|Maison                                  |630001 |498989   |
|Appartement                             |526412 |414292   |
|Local industriel. commercial ou assimilé|120706 |76149    |
+----------------------------------------+-------+---------+



## 3. Colonnes utiles : part de valeurs vides
Les colonnes utiles viennent de la question de Claire . Pour chacune : quel pourcentage de lignes est vide ?

In [4]:
UTILES = ["id_mutation", "date_mutation", "nature_mutation", "valeur_fonciere", "code_departement",
          "code_commune", "nom_commune", "id_parcelle", "type_local", "surface_reelle_bati",
          "nombre_pieces_principales", "surface_terrain", "longitude", "latitude"]

vides = df.agg(*[F.round(100 * F.avg(F.col(c).isNull().cast("int")), 1).alias(c) for c in UTILES]).first().asDict()
for col, pct in vides.items():
    print(f"{col:<28} {pct:>5} % vide")

bornes = df.agg(F.min("date_mutation"), F.max("date_mutation")).first()
print("Période couverte :", bornes[0], "->", bornes[1])

id_mutation                    0.0 % vide
date_mutation                  0.0 % vide
nature_mutation                0.0 % vide
valeur_fonciere                1.3 % vide
code_departement               0.0 % vide
code_commune                   0.0 % vide
nom_commune                    0.0 % vide
id_parcelle                    0.0 % vide
type_local                    39.5 % vide
surface_reelle_bati           39.5 % vide
nombre_pieces_principales     39.5 % vide
surface_terrain               31.4 % vide
longitude                      1.2 % vide
latitude                       1.2 % vide
Période couverte : 2025-01-01 -> 2025-12-31


## 4. Le prix : vide, nul, négatif ?


In [5]:
prix = df.withColumn("prix", F.expr("try_cast(valeur_fonciere AS double)"))
print("Prix vide ou non convertible :", prix.filter(F.col("prix").isNull()).count(), "lignes")
print("Prix égal à 0               :", prix.filter(F.col("prix") == 0).count(), "lignes")
print("Prix négatif                :", prix.filter(F.col("prix") < 0).count(), "lignes")

Prix vide ou non convertible : 50484 lignes
Prix égal à 0               : 0 lignes
Prix négatif                : 0 lignes


## 5. Structure des mutations
Combien de lignes par mutation ? Combien de lignes de logement (Maison ou Appartement) par mutation ? Cela vérifie l'hypothèse de  : *un logement = une ligne*.

In [6]:
lignes_par_mutation = df.groupBy("id_mutation").count()
lignes_par_mutation.groupBy(F.col("count").alias("lignes_par_mutation")).count().orderBy("lignes_par_mutation").show(10)

nb_log = (df.groupBy("id_mutation")
            .agg(F.sum(F.when(F.col("type_local").isin("Maison", "Appartement"), 1).otherwise(0)).alias("nb_logements")))
nb_log.groupBy("nb_logements").count().orderBy("nb_logements").show(10)

+-------------------+------+
|lignes_par_mutation| count|
+-------------------+------+
|                  1|530325|
|                  2|403123|
|                  3|207931|
|                  4| 98178|
|                  5| 40704|
|                  6| 30866|
|                  7| 16614|
|                  8| 12677|
|                  9|  8750|
|                 10|  6149|
+-------------------+------+
only showing top 10 rows
+------------+------+
|nb_logements| count|
+------------+------+
|           0|474361|
|           1|779461|
|           2|102315|
|           3| 13907|
|           4|  6770|
|           5|  2040|
|           6|  1875|
|           7|   736|
|           8|   702|
|           9|   329|
+------------+------+
only showing top 10 rows


## 6. Que contient la VEFA (vente en l'etat futur d'achèvement) 


In [7]:
vefa = df.filter(F.col("nature_mutation") == "Vente en l'état futur d'achèvement")

# (a) quels types de local dans la VEFA ?
(vefa.groupBy("type_local")
     .agg(F.count("*").alias("lignes"), F.countDistinct("id_mutation").alias("mutations"))
     .orderBy(F.desc("lignes"))
     .show(truncate=False))

# (b) combien de mutations VEFA contiennent au moins un logement (donc survivraient à R4a) ?
(vefa.groupBy("id_mutation")
     .agg(F.max(F.col("type_local").isin("Maison", "Appartement").cast("int")).alias("a_un_logement"))
     .groupBy("a_un_logement").count().show())

+----------------------------------------+------+---------+
|type_local                              |lignes|mutations|
+----------------------------------------+------+---------+
|NULL                                    |120442|42210    |
|Dépendance                              |12751 |8442     |
|Appartement                             |8143  |7745     |
|Local industriel. commercial ou assimilé|2137  |1136     |
|Maison                                  |400   |328      |
+----------------------------------------+------+---------+

+-------------+-----+
|a_un_logement|count|
+-------------+-----+
|         NULL|41318|
|            1| 8067|
|            0| 2597|
+-------------+-----+



** le prix au m² des ventes d'un seul logement, par nature de mutation. Si la VEFA est nettement plus chère, la mélanger fausserait le prix au m² des départements.

In [8]:
par_mutation = (df.withColumn("prix", F.expr("try_cast(valeur_fonciere AS double)"))
                  .withColumn("surf", F.expr("try_cast(surface_reelle_bati AS double)"))
                  .filter(F.col("type_local").isin("Maison", "Appartement"))
                  .groupBy("id_mutation", "nature_mutation")
                  .agg(F.count("*").alias("n"), F.max("prix").alias("prix"), F.max("surf").alias("surf"))
                  .filter((F.col("n") == 1) & (F.col("surf") >= 9) & (F.col("prix") > 0))
                  .withColumn("prix_m2", F.col("prix") / F.col("surf")))

(par_mutation.groupBy("nature_mutation")
             .agg(F.count("*").alias("ventes"), F.round(F.percentile_approx("prix_m2", 0.5), 0).alias("prix_m2_median"))
             .orderBy(F.desc("ventes"))
             .show(truncate=False))

+----------------------------------+------+--------------+
|nature_mutation                   |ventes|prix_m2_median|
+----------------------------------+------+--------------+
|Vente                             |757543|2636.0        |
|Vente en l'état futur d'achèvement|7846  |4666.0        |
|Adjudication                      |1834  |1433.0        |
|Echange                           |966   |1002.0        |
|Vente terrain à bâtir             |117   |2407.0        |
|Expropriation                     |4     |2333.0        |
+----------------------------------+------+--------------+



## 7. Vérifier l'hypothèse de : un logement = une ligne ?
Pour les mutations qui ont **deux** lignes de logement : sont-ce deux logements distincts (deux parcelles ou deux surfaces) ou le même logement répété ?

In [9]:
logements = df.filter(F.col("type_local").isin("Maison", "Appartement"))

deux = (logements.groupBy("id_mutation")
                 .agg(F.count("*").alias("n"),
                      F.countDistinct("id_parcelle").alias("parcelles"),
                      F.countDistinct("surface_reelle_bati").alias("surfaces"))
                 .filter(F.col("n") == 2))

(deux.groupBy((F.col("parcelles") == 1).alias("meme_parcelle"), (F.col("surfaces") == 1).alias("meme_surface"))
     .count().show())

+-------------+------------+-----+
|meme_parcelle|meme_surface|count|
+-------------+------------+-----+
|         true|       false|20911|
|         true|        true|76365|
|        false|       false| 4758|
|        false|        true|  281|
+-------------+------------+-----+



## 8. Les lignes « doubles » de logement : en quoi diffèrent-elles ?


In [10]:
exemples = ["2025-682654", "2025-712395", "2025-753325"]
cols = ["id_mutation", "id_parcelle", "type_local", "surface_reelle_bati",
        "code_nature_culture", "nature_culture", "nature_culture_speciale", "surface_terrain",
        "adresse_nom_voie", "longitude", "latitude"]
(logements.filter(F.col("id_mutation").isin(exemples))
          .select(*cols).orderBy("id_mutation", "nature_culture").show(10, truncate=False))

+-----------+--------------+----------+-------------------+-------------------+-------------------+-----------------------+---------------+------------------------------+-------------------+------------------+
|id_mutation|id_parcelle   |type_local|surface_reelle_bati|code_nature_culture|nature_culture     |nature_culture_speciale|surface_terrain|adresse_nom_voie              |longitude          |latitude          |
+-----------+--------------+----------+-------------------+-------------------+-------------------+-----------------------+---------------+------------------------------+-------------------+------------------+
|2025-682654|505850000A1140|Maison    |197                |S                  |sols               |NULL                   |500            |RTE DE HAUTTEVILLE            |-1.812262399999998 |49.473695965190856|
|2025-682654|505850000A1140|Maison    |197                |AG                 |terrains d'agrément|NULL                   |719            |RTE DE HAUTTEVILLE   

combien de lignes de logement sont de vrais doublons exacts (40 colonnes identiques), combien ne diffèrent que par la nature de culture / le terrain, et que devient le nombre de logements par mutation après déduplication ?

In [11]:
CLE = ["id_mutation", "id_parcelle", "type_local", "surface_reelle_bati", "nombre_pieces_principales", "lot1_numero"]

n_lignes = logements.count()
n_cle = logements.dropDuplicates(CLE).count()
n_exact = logements.dropDuplicates().count()
print(f"Lignes de logement                          : {n_lignes:,}".replace(",", " "))
print(f"Après déduplication exacte (40 colonnes)    : {n_exact:,}".replace(",", " "))
print(f"Après déduplication sur la clé du logement  : {n_cle:,}".replace(",", " "))

# Nombre de logements par mutation, en comptant les logements distincts
(logements.dropDuplicates(CLE).groupBy("id_mutation").count()
          .groupBy(F.col("count").alias("nb_logements")).count()
          .orderBy("nb_logements").show(10))

Lignes de logement                          : 1 156 413
Après déduplication exacte (40 colonnes)    : 1 103 793
Après déduplication sur la clé du logement  : 1 000 793
+------------+------+
|nb_logements| count|
+------------+------+
|           1|862015|
|           2| 33749|
|           3|  7088|
|           4|  3018|
|           5|  1479|
|           6|   870|
|           7|   477|
|           8|   316|
|           9|   186|
|          10|   134|
+------------+------+
only showing top 10 rows


In [12]:
logements = df.filter(F.col("type_local").isin("Maison", "Appartement"))

exemples = ["2025-682654", "2025-712395", "2025-753325"]
cols = ["id_mutation", "id_parcelle", "type_local", "surface_reelle_bati",
        "code_nature_culture", "nature_culture", "nature_culture_speciale", "surface_terrain",
        "adresse_nom_voie", "longitude", "latitude"]
(logements.filter(F.col("id_mutation").isin(exemples))
          .select(*cols).orderBy("id_mutation", "nature_culture").show(10, truncate=False))

+-----------+--------------+----------+-------------------+-------------------+-------------------+-----------------------+---------------+------------------------------+-------------------+------------------+
|id_mutation|id_parcelle   |type_local|surface_reelle_bati|code_nature_culture|nature_culture     |nature_culture_speciale|surface_terrain|adresse_nom_voie              |longitude          |latitude          |
+-----------+--------------+----------+-------------------+-------------------+-------------------+-----------------------+---------------+------------------------------+-------------------+------------------+
|2025-682654|505850000A1140|Maison    |197                |S                  |sols               |NULL                   |500            |RTE DE HAUTTEVILLE            |-1.812262399999998 |49.473695965190856|
|2025-682654|505850000A1140|Maison    |197                |AG                 |terrains d'agrément|NULL                   |719            |RTE DE HAUTTEVILLE   

In [13]:
CLE = ["id_mutation", "id_parcelle", "type_local", "surface_reelle_bati", "nombre_pieces_principales", "lot1_numero"]

n_lignes = logements.count()
n_cle = logements.dropDuplicates(CLE).count()
n_exact = logements.dropDuplicates().count()
print(f"Lignes de logement                          : {n_lignes:,}".replace(",", " "))
print(f"Après déduplication exacte (40 colonnes)    : {n_exact:,}".replace(",", " "))
print(f"Après déduplication sur la clé du logement  : {n_cle:,}".replace(",", " "))

# Nombre de logements par mutation, en comptant les logements distincts
(logements.dropDuplicates(CLE).groupBy("id_mutation").count()
          .groupBy(F.col("count").alias("nb_logements")).count()
          .orderBy("nb_logements").show(10))

Lignes de logement                          : 1 156 413
Après déduplication exacte (40 colonnes)    : 1 103 793
Après déduplication sur la clé du logement  : 1 000 793
+------------+------+
|nb_logements| count|
+------------+------+
|           1|862015|
|           2| 33749|
|           3|  7088|
|           4|  3018|
|           5|  1479|
|           6|   870|
|           7|   477|
|           8|   316|
|           9|   186|
|          10|   134|
+------------+------+
only showing top 10 rows


In [ ]:
spark.stop()